# L4 — Raggruppare, riassumere, disegnare (SOLUZIONI)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/soluzioni/L04_raggruppare/L04_soluzioni.ipynb)

## Promemoria

- **Shift + Invio** esegue una cella. **File → Salva una copia in Drive** per non perdere il lavoro.
- Per ogni passo trovi 🔎 **cosa cercare**. Gemini (✨) è disponibile, ma prova prima a capire cosa ti serve.
- ⏱️ Bloccato da più di 10 minuti? Chiedi.
- 👥 Si lavora in **gruppi da 3**, con tre ruoli che **ruotano a ogni passo**: chi **scrive** (condivide lo schermo e scrive il codice), chi **cerca** (Google, Gemini, dispense), chi **controlla** (il risultato ha senso? il numero torna?).
- Se fai un pasticcio: **Runtime → Riavvia ed esegui tutto**.

## Contesto

Il dataset è pulito (è il risultato della L3). Ora possiamo finalmente fare domande da economisti: come va il fatturato nel tempo? Quale regione ha le imprese più grandi? Quale settore è più produttivo?

**La domanda di oggi:** il numero che ottieni racconta davvero quello che sembra?

## 0. Preparazione

Esegui questa cella così com'è: carica il dataset pulito e la libreria per i grafici.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/imprese_pulito.xlsx")
df.shape

## Parte 1 — Nuove colonne

### 1. La produttività

Crea una nuova colonna `produttivita`: fatturato diviso addetti (migliaia di euro per addetto). Poi mostra le prime 5 righe per controllare.

🔎 *Da cercare:* `pandas nuova colonna` · `pandas dividere colonne`

In [ ]:
df["produttivita"] = df["fatturato"] / df["addetti"]
df[["fatturato", "addetti", "produttivita"]].head()

### 2. Controllala

Quanti valori mancanti ha `produttivita`? Sono di più di quelli di `fatturato`? Perché?

🔎 *Da cercare:* `pandas isna sum`

In [ ]:
print(df["fatturato"].isna().sum(), df["addetti"].isna().sum(), df["produttivita"].isna().sum())

*Risposta:* Sono di più: basta che manchi **uno** dei due valori (fatturato o addetti) perché il risultato sia mancante.

## Parte 2 — Raggruppare

### 3. Fatturato medio per anno

Calcola il fatturato **medio** per ogni anno.

🔎 *Da cercare:* `pandas groupby mean`

In [ ]:
df.groupby("anno")["fatturato"].mean()

### 4. Più statistiche insieme

Per ogni anno calcola media, mediana e numero di osservazioni del fatturato, in un'unica tabella.

🔎 *Da cercare:* `pandas groupby agg`

In [ ]:
df.groupby("anno")["fatturato"].agg(["mean", "median", "count"])

### 5. La regione più ricca?

Calcola il fatturato medio per regione e ordinalo dal più alto al più basso. Quale regione è prima?

🔎 *Da cercare:* `pandas sort_values ascending`

In [ ]:
df.groupby("regione")["fatturato"].mean().sort_values(ascending=False)

### 6. Ne sei sicuro?

Rifai la tabella per regione con **media**, **mediana** e **numero di imprese diverse**. La classifica regge? Guarda in particolare le prime due regioni.

🔎 *Da cercare:* `pandas agg named aggregation` · `pandas nunique`

In [ ]:
df.groupby("regione").agg(
    media=("fatturato", "mean"),
    mediana=("fatturato", "median"),
    imprese=("id_impresa", "nunique"),
).sort_values("media", ascending=False).head()

*Risposta:* La Liguria è prima per media solo grazie a un'impresa enorme (la sua mediana è nella norma). La Basilicata ha la mediana più alta, ma solo 2 imprese: troppo poche per dire qualcosa. Media, mediana e numero di osservazioni vanno guardati insieme.

### 7. I settori

Calcola la produttività **mediana** per settore, ordinata. Quale settore è il più produttivo?

🔎 *Da cercare:* `pandas groupby median`

In [ ]:
df.groupby("settore")["produttivita"].median().sort_values()

## Parte 3 — Disegnare

### 8. Nel tempo

Disegna un grafico a **linea** del fatturato **mediano** per anno. Cosa succede nel 2020?

🔎 *Da cercare:* `pandas plot line`

In [ ]:
df.groupby("anno")["fatturato"].median().plot(marker="o")
plt.show()

*Risposta:* Nel 2020 il fatturato mediano crolla (circa un quarto in meno) e nel 2021 torna ai livelli precedenti: è lo shock della pandemia.

### 9. Per categoria

Disegna un grafico a **barre orizzontali** della produttività mediana per settore, ordinate.

🔎 *Da cercare:* `pandas plot barh`

In [ ]:
df.groupby("settore")["produttivita"].median().sort_values().plot(kind="barh")
plt.show()

### 10. La distribuzione

Disegna un **istogramma** del fatturato. Che forma ha? Ti spiega perché media e mediana sono così diverse?

🔎 *Da cercare:* `pandas plot hist bins`

In [ ]:
df["fatturato"].plot(kind="hist", bins=50)
plt.show()
print(df["fatturato"].mean(), df["fatturato"].median())

*Risposta:* È molto asimmetrica: tantissime imprese piccole e poche molto grandi. Le poche grandi tirano su la media (circa 2.600) ben sopra la mediana (circa 1.400).

## Domanda finale

Un giornalista ti chiede: *"Qual è la regione con le imprese più ricche?"* Scrivi una frase che gli diresti, e una che **non** gli diresti.

*Risposta:* Direi: "le differenze tra regioni sono piccole; per fatturato mediano spiccano Umbria e Sardegna, ma su pochi dati". Non direi: "la Liguria è la regione più ricca" (lo è solo in media, per un'impresa sola) né "la Basilicata" (2 imprese).

---
## ⭐ Bonus (per chi ha finito)

### B1. Un grafico presentabile

Riprendi il grafico nel tempo e aggiungi titolo, etichette degli assi con le unità di misura, e salvalo come immagine `fatturato_anno.png`.

🔎 *Da cercare:* `matplotlib title xlabel ylabel` · `matplotlib savefig`

In [ ]:
df.groupby("anno")["fatturato"].median().plot(marker="o")
plt.title("Fatturato mediano delle imprese, 2019-2023")
plt.xlabel("Anno")
plt.ylabel("Migliaia di euro")
plt.savefig("fatturato_anno.png", dpi=150, bbox_inches="tight")
plt.show()

### B2. Una tabella a doppia entrata

Costruisci una tabella con le regioni sulle righe, gli anni sulle colonne e il fatturato mediano nelle celle.

🔎 *Da cercare:* `pandas pivot_table`

In [ ]:
df.pivot_table(index="regione", columns="anno", values="fatturato", aggfunc="median").round(0)

### B3. Scala logaritmica

Rifai l'istogramma del fatturato con l'asse orizzontale in scala logaritmica. Cosa cambia?

🔎 *Da cercare:* `numpy log` · `matplotlib xscale log`

In [ ]:
import numpy as np
np.log10(df["fatturato"]).plot(kind="hist", bins=50)
plt.xlabel("log10(fatturato)")
plt.show()

---
## Compito a casa

- [Kaggle Learn – Pandas](https://www.kaggle.com/learn/pandas), lezione **Grouping and Sorting** (~20 min)
- Facoltativo: [QuantEcon – Matplotlib](https://python-programming.quantecon.org/matplotlib.html)